# LiteLLM Python SDK — Introduction

> **Description:** This notebook is a hands-on tour of why many AI engineers reach for [LiteLLM](https://docs.litellm.ai/) instead of wiring up each provider's SDK by hand.

- https://www.litellm.ai/  
- https://docs.litellm.ai/docs/learn  
- https://docs.litellm.ai/docs/  

LiteLLM lets you call **100+ LLM providers** — OpenAI, Anthropic, Azure, Bedrock, Gemini, Ollama, and more — through **one consistent function**:


```python
litellm.completion(model=..., messages=...)
```

Instead of learning a different SDK for every provider...

```python
openai.OpenAI().chat.completions.create(...)      # OpenAI's way
anthropic.Anthropic().messages.create(...)        # Anthropic's way
genai.Client().models.generate_content(...)       # Google's way
```

...you write the call **once**, using OpenAI's familiar `messages=[{"role": ..., "content": ...}]` format, and just swap the `model` string to switch providers.

## What You'll Learn

| # | Benefit |
|---|---|
| 1 | Unified interface across providers |
| 2 | Consistent response shape |
| 3 | Built-in cost tracking |
| 4 | Built-in reliability (fallbacks) |
| 5 | Uniform streaming |
| 6 | Native async support |



## Setup

Before we dive in, let's load your API keys from `.env`.

**Requires:** `OPENAI_API_KEY`
**Optional:** `ANTHROPIC_API_KEY` — a couple of cells use it for a side-by-side comparison, and will politely skip themselves if it's missing.

> ⚠️ **Run this notebook from the `part_2_concepts/` folder** — that's what makes the relative `.env` path work, matching the rest of the samples here.

In [2]:
import logging
import os
import time
import warnings

from dotenv import load_dotenv

load_dotenv(".env", override=True)

HAS_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY"))
HAS_ANTHROPIC_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))

print(f"OPENAI_API_KEY set:    {HAS_OPENAI_KEY}")
print(f"ANTHROPIC_API_KEY set: {HAS_ANTHROPIC_KEY}")

OPENAI_API_KEY set:    True
ANTHROPIC_API_KEY set: False


# Supress Error and Warning messages

In [3]:
import logging
import litellm

logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)  # hide the internal fallback error traceback
litellm.suppress_debug_info = True  # hide the "Give Feedback / Get Help" footer

# LiteLLM runs a background asyncio task for logging callbacks; when Python's
# garbage collector reclaims it (timing varies, can surface in any later cell)
# asyncio logs a noisy "Task was destroyed but it is pending!" message. Silence
# it notebook-wide, up front, rather than chasing which cell it lands on.
logging.getLogger("asyncio").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message=".*coroutine.*was never awaited")

## 1. One Interface, 100+ Providers

Without LiteLLM, calling OpenAI vs. Anthropic means juggling two different SDKs, two different client objects, and two different call shapes:

```python
# OpenAI SDK
from openai import OpenAI
OpenAI().chat.completions.create(model="gpt-5.4-nano", messages=messages)

# Anthropic SDK
from anthropic import Anthropic
Anthropic().messages.create(model="claude-sonnet-5", max_tokens=1024, messages=messages)
```

With LiteLLM, it's the **same function call** for both — only the `model` string (prefixed with the provider name) changes.

In [4]:
from litellm import completion

messages = [{"role": "user", "content": "In one short sentence, what is LiteLLM?"}]

openai_response = completion(model="openai/gpt-5.4-nano", messages=messages)
print("OpenAI:", openai_response.choices[0].message.content)

OpenAI: LiteLLM is an open-source library that lets you connect to multiple LLM providers (like OpenAI, Anthropic, and others) through a single, unified API.


In [5]:
# Exact same function, exact same `messages` format — only the model string changes.
if HAS_ANTHROPIC_KEY:
    anthropic_response = completion(model="anthropic/claude-sonnet-5", messages=messages)
    print("Anthropic:", anthropic_response.choices[0].message.content)


## 2. Consistent Response Shape

Every provider normally hands you back a differently-shaped object — OpenAI's `ChatCompletion`, Anthropic's `Message`, and so on — so switching providers usually means rewriting your parsing code too.

LiteLLM always returns an **OpenAI-shaped `ModelResponse`**, no matter the provider:

- `response.choices[0].message.content` → the text
- `response.usage.total_tokens` → token counts
- `response.model` → which model actually answered

Your parsing code stays exactly the same, forever.

In [6]:
print("content:      ", openai_response.choices[0].message.content[:60], "...")
print("role:         ", openai_response.choices[0].message.role)
print("finish_reason:", openai_response.choices[0].finish_reason)
print("usage:        ", openai_response.usage)
print("model:        ", openai_response.model)

content:       LiteLLM is an open-source library that lets you connect to m ...
role:          assistant
finish_reason: stop
usage:         Usage(completion_tokens=38, prompt_tokens=17, total_tokens=55, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None))
model:         gpt-5.4-nano-2026-03-17


In [ ]:
# print(openai_response)
# print(openai_response._hidden_params)


ModelResponse(id='chatcmpl-E4Zi3DXQc3iAkOoMVa64zSV2xQMKs', created=1784759287, model='gpt-5.4-nano-2026-03-17', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='LiteLLM is an open-source library that lets you connect to multiple LLM providers (like OpenAI, Anthropic, and others) through a single, unified API.', role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None}, annotations=[]), provider_specific_fields={})], usage=Usage(completion_tokens=38, prompt_tokens=17, total_tokens=55, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None)), moderation=None, serv

## 3. Built-in Cost Tracking

Normally, tracking the dollar cost of a call means maintaining your *own* price table for every provider and model — and updating it every time prices change.

LiteLLM ships that pricing data for 100+ models and computes the cost for you with a single call: `completion_cost()`.

In [7]:
from litellm import completion_cost

cost = completion_cost(completion_response=openai_response)
print(f"Cost of that call: ${cost:.8f}")

Cost of that call: $0.00005090


## 4. Built-in Reliability via Fallbacks

Without LiteLLM, handling "model X is down or rate-limited, retry with model Y" means hand-rolling your own try/except ladder.

LiteLLM supports this natively — just pass a `fallbacks=[...]` list. If the primary `model` fails, LiteLLM automatically retries with the next model in line. No custom retry logic required.

Below, the primary model name is *intentionally* invalid so you can watch the automatic fallback kick in. By default LiteLLM logs the failed attempt as a noisy error/traceback; the cell below silences that internal log so only the final result prints.

In [8]:
fallback_response = completion(
    model="openai/gpt-does-not-exist",
    messages=[{"role": "user", "content": "Say 'fallback worked' and nothing else."}],
    fallbacks=["openai/gpt-5.4-nano"],
)
print("Response came from model:", fallback_response.model)
print("Content:", fallback_response.choices[0].message.content)

Response came from model: gpt-5.4-nano-2026-03-17
Content: fallback worked


## 5. Uniform Streaming

`stream=True` behaves the same way — an iterator of OpenAI-shaped chunks with `.choices[0].delta.content` — no matter which provider is behind `model`.

In [9]:
stream = completion(
    model="openai/gpt-5.4-nano",
    messages=[{"role": "user", "content": "Count from 1 to 100 and output lines in format Counting #1, new line separated."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

Counting #1  
Counting #2  
Counting #3  
Counting #4  
Counting #5  
Counting #6  
Counting #7  
Counting #8  
Counting #9  
Counting #10  
Counting #11  
Counting #12  
Counting #13  
Counting #14  
Counting #15  
Counting #16  
Counting #17  
Counting #18  
Counting #19  
Counting #20  
Counting #21  
Counting #22  
Counting #23  
Counting #24  
Counting #25  
Counting #26  
Counting #27  
Counting #28  
Counting #29  
Counting #30  
Counting #31  
Counting #32  
Counting #33  
Counting #34  
Counting #35  
Counting #36  
Counting #37  
Counting #38  
Counting #39  
Counting #40  
Counting #41  
Counting #42  
Counting #43  
Counting #44  
Counting #45  
Counting #46  
Counting #47  
Counting #48  
Counting #49  
Counting #50  
Counting #51  
Counting #52  
Counting #53  
Counting #54  
Counting #55  
Counting #56  
Counting #57  
Counting #58  
Counting #59  
Counting #60  
Counting #61  
Counting #62  
Counting #63  
Counting #64  
Counting #65  
Counting #66  
Counting #67  
Coun

## 6. Native Async Support

`litellm.acompletion()` is the async twin of `completion()`, so you can fire off multiple calls concurrently with `asyncio.gather` — useful for calling several models (or several prompts) in parallel instead of sequentially.

In [10]:
import asyncio

from litellm import acompletion

prompts = [
    "Name one benefit of async code, in 5 words or fewer.",
    "Name one risk of async code, in 5 words or fewer.",
]

def get_tasks(task_prompts):
    tasks = [
        acompletion(model="openai/gpt-5.4-nano", messages=[{"role": "user", "content": p}])
        for p in prompts
    ]
    return tasks

async def ask_all(tasks):    
    return await asyncio.gather(*tasks)

tasks = get_tasks(prompts);
# print(tasks)

start = time.time()
results = await ask_all(tasks)
elapsed = time.time() - start

for prompt, result in zip(prompts, results):
    print(f"- {prompt}\n  -> {result.choices[0].message.content}")
print(f"\n{len(prompts)} calls completed concurrently in {elapsed:.2f}s")

- Name one benefit of async code, in 5 words or fewer.
  -> Improved responsiveness during I/O.
- Name one risk of async code, in 5 words or fewer.
  -> Race conditions between tasks.

2 calls completed concurrently in 0.84s


## Summary

| Benefit | Without LiteLLM | With LiteLLM |
|---|---|---|
| Switching providers | Rewrite SDK calls + response parsing | Change one `model` string |
| Response parsing | Different shape per provider | Always OpenAI-shaped `choices[0].message.content` |
| Cost tracking | Maintain your own price table | `completion_cost(response)` |
| Reliability | Hand-rolled retry/fallback loops | `fallbacks=[...]` parameter |
| Streaming | Provider-specific chunk formats | Same `delta.content` pattern everywhere |
| Async / concurrency | Per-SDK async client setup | `acompletion()` drop-in async twin |

LiteLLM is a routing/normalization layer over each provider's *real* API — it still requires that provider's own API key.
